Install and Import packages

Allow Pandas dataframes to interface directly with a Postgres database engine

In [120]:
pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


Safely read environment config variables from local files

In [121]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


Handles HTTP requests to pull raw data from the USAJobs API

In [122]:
import requests


Data Manipulation

In [123]:
import pandas as pd

Engine creator used to establish the target SQL connection route

In [124]:
from sqlalchemy import create_engine

Handles dates and timestamps manipulation

In [125]:
from datetime import datetime

NLP tool to evaluate string similarities

In [126]:
from sklearn.feature_extraction.text import TfidfVectorizer

Tool for text cleaning patterns

In [127]:
import re

System library to read environment variables from the OS

In [128]:
import os

Imports the key-value secrets from a local .env file

In [129]:
from dotenv import load_dotenv

Extract Layer

In [130]:
roles = [
    "data analyst",
    "data scientist",
    "data engineer",
    "business analyst",
    "machine learning engineer",
    "analytics engineer",
    "data specialist",
    "research analyst",
    "business intelligence analyst",
    "data architect",
    "data warehouse engineer",
    "database administrator",
    
]

Loads hidden API keys and passwords from the .env file into Python's environment

In [131]:
load_dotenv()

True

Fetch my secret USAJobs API credentials securely from the system environment variables

In [132]:
USER_AGENT = os.getenv("USAJOBS_USER_AGENT")
AUTH_KEY = os.getenv("USAJOBS_AUTH_KEY")

Set up the API endpoint URL and the required security headers for authentication

In [133]:
url = "https://data.usajobs.gov/api/search"

headers = {
    "User-Agent": USER_AGENT,
    "Authorization-Key": AUTH_KEY
}

In [134]:
# Initialize an empty master list to store all collected job dictionaries
all_jobs = []
#Loop through my defined role list
for role in roles:

    page = 1 
    #Infinite loop to handle API pagination (continues until no more jobs are found)
    while True: 
     params = {
            "PositionTitle": role,
            "ResultsPerPage": 100,
            "Page": page,
            "Fields": "full",
          
        }
     # Send the GET request
     response = requests.get(url, headers=headers, params=params)
     data = response.json()
     # Extract the list of job postings. Return an empty list if none exist
     jobs = data.get("SearchResult", {}).get("SearchResultItems", [])
     # If the current page returns zero jobs, break the while loop and move to the next role
     if not jobs:
       break
     # Loop through each job
     for job in jobs:
            obj = job["MatchedObjectDescriptor"]
    
             # Get the is_remote data. As it's in the complex nested json format, I need to break it down
            remote_val = obj.get("RemoteIndicator")
            if str(remote_val).lower() == 'true':
                is_remote_val = True
            elif str(remote_val).lower() == 'false':
                is_remote_val = False
            else:
                is_remote_val = False 
        
            all_jobs.append({
                "role": role,
                "title": obj.get("PositionTitle"),
                "location": obj.get("PositionLocationDisplay"),
                "description": obj.get("UserArea", {}).get("Details", {}).get("JobSummary", ""),
                "posted_date": obj.get("PublicationStartDate"),
                "close_date": obj.get("ApplicationCloseDate"),
                "job_grade": obj.get("JobGrade"),
                "salary_min": obj.get("PositionRemuneration", [{}])[0].get("MinimumRange"),
                "salary_max": obj.get("PositionRemuneration", [{}])[0].get("MaximumRange"),
                "employment_type": obj.get("PositionSchedule", [{}])[0] if isinstance(obj.get("PositionSchedule"), list) else obj.get("PositionSchedule"),
                
                "is_remote": is_remote_val,  
            
                
            
            })
       
     # Fetch the next batch
     page += 1

 


In [135]:
df = pd.DataFrame(all_jobs)
print(df.shape)
df.head()

(5666, 11)


,role,title,location,description,posted_date,close_date,job_grade,salary_min,salary_max,employment_type,is_remote
0,data analyst,Data Analyst,"Arlington, Virginia",*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11T11:49:11.8070,2026-06-22T23:59:59.9970,[{'Code': 'GG'}],70623,158322,"{'Name': '', 'Code': '1'}",False
1,data analyst,Program Analyst (Data Analyst),Multiple Locations,As a Program Analyst (Data Analyst) at the GS-...,2026-06-10T00:00:00.0000,2026-06-16T23:59:59.9970,[{'Code': 'GS'}],63795,99404,"{'Name': '', 'Code': '1'}",False
2,data analyst,Data Engineer,"Washington, District of Columbia",DDI Data Engineers solve critical data challen...,2025-10-01T00:00:00.0000,2026-09-30T23:59:59.9970,[{'Code': 'GS'}],74584,156755,"{'Name': '', 'Code': '1'}",False
3,data analyst,DATA SCIENTIST,Multiple Locations,The PALACE Acquire Program offers you a perman...,2025-09-29T00:00:00.0000,2026-09-28T23:59:59.9970,[{'Code': 'GS'}],49960,99314,"{'Name': '', 'Code': '1'}",False
4,data analyst,Data Scientist,"Woodlawn, Maryland",IT Specialist (Data Scientist) positions are b...,2026-02-06T00:00:00.0000,2026-08-05T23:59:59.9970,[{'Code': 'GS'}],143913,187093,{'Name': 'Flextime and Alternate Work Schedule...,False


In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5666 entries, 0 to 5665
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   role             5666 non-null   object
 1   title            5666 non-null   object
 2   location         5666 non-null   object
 3   description      5666 non-null   object
 4   posted_date      5666 non-null   object
 5   close_date       5666 non-null   object
 6   job_grade        5666 non-null   object
 7   salary_min       5666 non-null   object
 8   salary_max       5666 non-null   object
 9   employment_type  5666 non-null   object
 10  is_remote        5666 non-null   bool  
dtypes: bool(1), object(10)
memory usage: 448.3+ KB


**Transform Layer**

Drop Duplicates

In [137]:
df = df.drop_duplicates(subset=["title", "location", "description", "posted_date"])

In [138]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2829 entries, 0 to 5299
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   role             2829 non-null   object
 1   title            2829 non-null   object
 2   location         2829 non-null   object
 3   description      2829 non-null   object
 4   posted_date      2829 non-null   object
 5   close_date       2829 non-null   object
 6   job_grade        2829 non-null   object
 7   salary_min       2829 non-null   object
 8   salary_max       2829 non-null   object
 9   employment_type  2829 non-null   object
 10  is_remote        2829 non-null   bool  
dtypes: bool(1), object(10)
memory usage: 245.9+ KB


In [139]:
df.isnull().sum()/len(df)

role               0.0
title              0.0
location           0.0
description        0.0
posted_date        0.0
close_date         0.0
job_grade          0.0
salary_min         0.0
salary_max         0.0
employment_type    0.0
is_remote          0.0
dtype: float64

Transform Job Grade

In [140]:
df["job_grade"] = df["job_grade"].apply(
    lambda x: x[0]["Code"] if isinstance(x, list) and len(x) > 0 else x
)

Transform Employment Type

In [141]:
df["employment_type"] = df["employment_type"].apply(
    lambda x: x.get("Code") if isinstance(x, dict) else x
)

Map Employement Type

In [142]:
e_map = {"1": "Full-Time",
       "2": "Part-Time",
       "3": "Shift Work",
       "4": "Intermittent",
       "5": "Job Share",
       "6": "On-Call"}

In [143]:
df['employment_type']=df['employment_type'].map(e_map).fillna(df['employment_type'])

Change Date Format

In [144]:
df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce").dt.date
df["close_date"] = pd.to_datetime(df["close_date"], errors="coerce").dt.date

Transform Location

In [145]:
us_states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", 
    "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", 
    "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", 
    "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", 
    "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio", 
    "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota", 
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", 
    "Wisconsin", "Wyoming", "District of Columbia"
]

In [146]:
# split location into city and state
df[["city","state"]]=df["location"].str.split(",",n=1,expand=True)
# remove spaces
df["city"]=df["city"].str.strip()
df["state"]=df["state"].str.strip()
# Fill in the rows that say "Multiple Locations"
df["state"]=df["state"].fillna("Multiple Locations")
df["city"]=df["city"].replace("Multiple Locations", "Various Cities")
#Filter out jobs that are not in the US
df=df[df["state"].isin(us_states)]

In [147]:
df.drop(columns="location")

,role,title,description,posted_date,close_date,job_grade,salary_min,salary_max,employment_type,is_remote,city,state
0,data analyst,Data Analyst,*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623,158322,Full-Time,False,Arlington,Virginia
2,data analyst,Data Engineer,DDI Data Engineers solve critical data challen...,2025-10-01,2026-09-30,GS,74584,156755,Full-Time,False,Washington,District of Columbia
4,data analyst,Data Scientist,IT Specialist (Data Scientist) positions are b...,2026-02-06,2026-08-05,GS,143913,187093,Full-Time,False,Woodlawn,Maryland
8,data analyst,Data Scientist,This vacancy is being filled via the Direct-Hi...,2026-06-04,2026-06-18,GG,102415,133142,Full-Time,False,Suitland,Maryland
11,data analyst,DATA SCIENTIST,You will serve as a Data Scientist in the Anal...,2026-06-12,2026-06-16,GS,121785,158322,Full-Time,False,Patuxent River,Maryland
...,...,...,...,...,...,...,...,...,...,...,...,...
5280,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,Marine Corps Community Services (MCCS) is look...,2026-05-18,2026-06-15,NF,16,16,Part-Time,False,Kaneohe,Hawaii
5281,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,Marine Corps Community Services (MCCS) is look...,2026-06-11,2026-07-09,NF,16,16,Part-Time,False,Kaneohe,Hawaii
5292,data warehouse engineer,"Material Handler Supervisor, (Warehouse Worker...",Corrections professionals who foster a humane ...,2026-06-03,2026-06-17,WS,28.79,33.6,Full-Time,False,Glenville,West Virginia
5295,data warehouse engineer,Materials Handler Supervisor (Warehouse Worker...,Corrections professionals who foster a humane ...,2026-06-04,2026-06-26,WS,37.07,43.25,Full-Time,False,Marion,Illinois


In [148]:
order=["role", "title", "city", "state", "description", "posted_date", "close_date", "job_grade", "salary_min", "salary_max", "employment_type", "is_remote"]

In [149]:
df=df[order]

In [150]:
df

,role,title,city,state,description,posted_date,close_date,job_grade,salary_min,salary_max,employment_type,is_remote
0,data analyst,Data Analyst,Arlington,Virginia,*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623,158322,Full-Time,False
2,data analyst,Data Engineer,Washington,District of Columbia,DDI Data Engineers solve critical data challen...,2025-10-01,2026-09-30,GS,74584,156755,Full-Time,False
4,data analyst,Data Scientist,Woodlawn,Maryland,IT Specialist (Data Scientist) positions are b...,2026-02-06,2026-08-05,GS,143913,187093,Full-Time,False
8,data analyst,Data Scientist,Suitland,Maryland,This vacancy is being filled via the Direct-Hi...,2026-06-04,2026-06-18,GG,102415,133142,Full-Time,False
11,data analyst,DATA SCIENTIST,Patuxent River,Maryland,You will serve as a Data Scientist in the Anal...,2026-06-12,2026-06-16,GS,121785,158322,Full-Time,False
...,...,...,...,...,...,...,...,...,...,...,...,...
5280,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,Kaneohe,Hawaii,Marine Corps Community Services (MCCS) is look...,2026-05-18,2026-06-15,NF,16,16,Part-Time,False
5281,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,Kaneohe,Hawaii,Marine Corps Community Services (MCCS) is look...,2026-06-11,2026-07-09,NF,16,16,Part-Time,False
5292,data warehouse engineer,"Material Handler Supervisor, (Warehouse Worker...",Glenville,West Virginia,Corrections professionals who foster a humane ...,2026-06-03,2026-06-17,WS,28.79,33.6,Full-Time,False
5295,data warehouse engineer,Materials Handler Supervisor (Warehouse Worker...,Marion,Illinois,Corrections professionals who foster a humane ...,2026-06-04,2026-06-26,WS,37.07,43.25,Full-Time,False


Extract key skills

In [152]:
df_1=df.copy()

In [153]:
skill_filter = [
    "python", "sql", "tableau", "power bi", "excel",
    "aws", "spark", "snowflake", "machine learning"
]

In [154]:
texts = df_1["description"].fillna("").str.lower()

In [155]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
X = vectorizer.fit_transform(df_1["description"].fillna("").str.lower())

In [156]:
feature_names = vectorizer.get_feature_names_out()

In [157]:
# Get top keywords. I try to give those skill filter a boost
def get_top_keywords(row, top_n=10):
    row = row.toarray().flatten()

    scored_words = []

    for i, score in enumerate(row):
        word = feature_names[i]

        if score > 0:
            # base TF-IDF score
            final_score = score

            # BOOST if it's a skill
            if word in skill_filter:
                final_score *= 3

            scored_words.append((word, final_score))

    # sort by final score
    scored_words.sort(key=lambda x: x[1], reverse=True)

    return [word for word, score in scored_words[:top_n]]

In [158]:
df_1["top_keywords"] = [
    get_top_keywords(X[i]) for i in range(X.shape[0])
]

In [159]:
df_1

,role,title,city,state,description,posted_date,close_date,job_grade,salary_min,salary_max,employment_type,is_remote,top_keywords
0,data analyst,Data Analyst,Arlington,Virginia,*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623,158322,Full-Time,False,"[united, states, service, commission, inspecto..."
2,data analyst,Data Engineer,Washington,District of Columbia,DDI Data Engineers solve critical data challen...,2025-10-01,2026-09-30,GS,74584,156755,Full-Time,False,"[data, delivering, analyzing, building, advanc..."
4,data analyst,Data Scientist,Woodlawn,Maryland,IT Specialist (Data Scientist) positions are b...,2026-02-06,2026-08-05,GS,143913,187093,Full-Time,False,"[civil, appointments, new, authority, hire, se..."
8,data analyst,Data Scientist,Suitland,Maryland,This vacancy is being filled via the Direct-Hi...,2026-06-04,2026-06-18,GG,102415,133142,Full-Time,False,"[census, suitland, bureau, rail, green, maryla..."
11,data analyst,DATA SCIENTIST,Patuxent River,Maryland,You will serve as a Data Scientist in the Anal...,2026-06-12,2026-06-16,GS,121785,158322,Full-Time,False,"[analytics, river, maryland, fleet, scientist,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5280,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,Kaneohe,Hawaii,Marine Corps Community Services (MCCS) is look...,2026-05-18,2026-06-15,NF,16,16,Part-Time,False,"[mccs, corps, marine, community, team, brighte..."
5281,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,Kaneohe,Hawaii,Marine Corps Community Services (MCCS) is look...,2026-06-11,2026-07-09,NF,16,16,Part-Time,False,"[mccs, corps, marine, community, team, brighte..."
5292,data warehouse engineer,"Material Handler Supervisor, (Warehouse Worker...",Glenville,West Virginia,Corrections professionals who foster a humane ...,2026-06-03,2026-06-17,WS,28.79,33.6,Full-Time,False,"[humane, reentry, corrections, foster, profess..."
5295,data warehouse engineer,Materials Handler Supervisor (Warehouse Worker...,Marion,Illinois,Corrections professionals who foster a humane ...,2026-06-04,2026-06-26,WS,37.07,43.25,Full-Time,False,"[humane, reentry, corrections, foster, profess..."


=> This didn't work, as the keywords extracted were not important skills

Mapping method

In [160]:
'''
skills = [
    # programming
    "python", "r", "sql", "java",

    # data tools
    "excel", "tableau", "power bi",

    # cloud / engineering
    "aws", "azure", "snowflake", "spark", "databricks",

    # analytics / ml
    "data analysis", "statistics", "machine learning",
    "regression", "forecasting",

    # business
    "data visualization", "reporting", "etl"
]
'''


'\nskills = [\n    # programming\n    "python", "r", "sql", "java",\n\n    # data tools\n    "excel", "tableau", "power bi",\n\n    # cloud / engineering\n    "aws", "azure", "snowflake", "spark", "databricks",\n\n    # analytics / ml\n    "data analysis", "statistics", "machine learning",\n    "regression", "forecasting",\n\n    # business\n    "data visualization", "reporting", "etl"\n]\n'

In [161]:
'''
df["skills_found"] = df["description"].str.lower().apply(
    lambda text: [
        skill for skill in skills
        if re.search(rf"\b{re.escape(skill)}\b", text) # make the keyword matching more precise
    ]
)
'''


'\ndf["skills_found"] = df["description"].str.lower().apply(\n    lambda text: [\n        skill for skill in skills\n        if re.search(rf"\x08{re.escape(skill)}\x08", text) # make the keyword matching more precise\n    ]\n)\n'

In [162]:
'''
df["skills_found"] = df["skills_found"].apply(
    lambda x: x if isinstance(x, list) else []
)
'''

'\ndf["skills_found"] = df["skills_found"].apply(\n    lambda x: x if isinstance(x, list) else []\n)\n'

=> This didn't work as these skills may not appear in all jobs

**Load**

In [163]:
df.to_csv("job_skills.csv", index=False)

In [164]:
df.columns = df.columns.str.lower().str.strip()
df = df.reset_index(drop=True)



In [165]:
df

,role,title,city,state,description,posted_date,close_date,job_grade,salary_min,salary_max,employment_type,is_remote
0,data analyst,Data Analyst,Arlington,Virginia,*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623,158322,Full-Time,False
1,data analyst,Data Engineer,Washington,District of Columbia,DDI Data Engineers solve critical data challen...,2025-10-01,2026-09-30,GS,74584,156755,Full-Time,False
2,data analyst,Data Scientist,Woodlawn,Maryland,IT Specialist (Data Scientist) positions are b...,2026-02-06,2026-08-05,GS,143913,187093,Full-Time,False
3,data analyst,Data Scientist,Suitland,Maryland,This vacancy is being filled via the Direct-Hi...,2026-06-04,2026-06-18,GG,102415,133142,Full-Time,False
4,data analyst,DATA SCIENTIST,Patuxent River,Maryland,You will serve as a Data Scientist in the Anal...,2026-06-12,2026-06-16,GS,121785,158322,Full-Time,False
...,...,...,...,...,...,...,...,...,...,...,...,...
2210,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,Kaneohe,Hawaii,Marine Corps Community Services (MCCS) is look...,2026-05-18,2026-06-15,NF,16,16,Part-Time,False
2211,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,Kaneohe,Hawaii,Marine Corps Community Services (MCCS) is look...,2026-06-11,2026-07-09,NF,16,16,Part-Time,False
2212,data warehouse engineer,"Material Handler Supervisor, (Warehouse Worker...",Glenville,West Virginia,Corrections professionals who foster a humane ...,2026-06-03,2026-06-17,WS,28.79,33.6,Full-Time,False
2213,data warehouse engineer,Materials Handler Supervisor (Warehouse Worker...,Marion,Illinois,Corrections professionals who foster a humane ...,2026-06-04,2026-06-26,WS,37.07,43.25,Full-Time,False


In [166]:
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DATABASE = os.getenv("DB_NAME")

In [167]:
database_url = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(database_url)

Check if the data already existed

In [168]:
try:
    df_existing = pd.read_sql("SELECT * FROM job_skills", engine)
except Exception:
    df_existing = pd.DataFrame()

In [169]:
df_combined = pd.concat([df_existing, df], ignore_index=True)


In [170]:
df_combined = df_combined.drop_duplicates(
    subset=["title", "city", "state", "description", "posted_date"], 
    keep="last"
)

In [171]:
df_combined.to_sql("job_skills", engine, if_exists="replace", index=False)

183